# **Exercicio Duelo de Modelos 4**

Nesta tarefa, vocês irão criar o seu próprio duelo de modelos, com o objetivo de superar os resultados apresentados em aula. O desafio é alcançar um desempenho superior ao que obtivemos, e para isso, será necessário aplicar todas as melhorias que vocês aprenderam ao longo dos módulos, utilizando a base de dados do Titanic.

https://www.kaggle.com/c/titanic

**1. Escolha do Modelo:**
Selecione um dos modelos que foram explorados nos duelos de modelos ao longo do curso. Pode ser SVM, Random Forest, XGBoost, ou qualquer outro que tenhamos abordado.

Modelos escolhidos: random Forest e XGBOOST

## Tratamento dos dados de treinamento

Excluir categorias não relevantes, como tickete nome, avaliar tipos de dados, erros de digitação 
a categoria cabine pode ser alterada para binário, pois aqueles que não estavam na primeira classe não tinham cabine, enquanto os outros tinham, então é o suficiente diferenciar os dados entre tem cabine não tem

In [ ]:
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np

In [71]:
xy_train = pd.read_csv('train.csv')
x_test = pd.read_csv('test.csv')
xy_train.drop(['Ticket', 'Name', 'PassengerId'], axis=1, inplace=True)
x_test.drop(['Ticket', 'Name', 'PassengerId'], axis=1, inplace=True)
print(xy_train.columns,"\n", x_test.columns)

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin',
       'Embarked'],
      dtype='object') 
 Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked'], dtype='object')


In [72]:

xy_train['Cabin'] = xy_train['Cabin'].notna().astype(int)
xy_train.fillna({'Cabin':0}, inplace=True)

x_test['Cabin'] = x_test['Cabin'].notna().astype(int)
x_test.fillna({'Cabin':0}, inplace=True)


print(xy_train['Cabin'].unique(),"\n",x_test['Cabin'].unique())


[0 1] 
 [0 1]


corrigir as variaveis de fare e sex para numerico, e corrigir os nulos da idade

In [73]:
xy_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Cabin     891 non-null    int64  
 8   Embarked  889 non-null    object 
dtypes: float64(2), int64(5), object(2)
memory usage: 62.8+ KB


In [74]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
xy_train['Sex'] = label_encoder.fit_transform(xy_train['Sex'])
x_test['Sex'] = label_encoder.fit_transform(x_test['Sex'])
# Oneenconder
xy_train = pd.get_dummies(xy_train, columns=['Embarked'], prefix='Embarked')
x_test = pd.get_dummies(x_test, columns=['Embarked'], prefix='Embarked')
print(xy_train.dtypes)

Survived        int64
Pclass          int64
Sex             int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Cabin           int64
Embarked_C       bool
Embarked_Q       bool
Embarked_S       bool
dtype: object


In [75]:
xy_train['Age'] = xy_train['Age'].fillna(xy_train['Age'].mean())
x_test['Age'] = x_test['Age'].fillna(x_test['Age'].mean())


## Busca por outliers e balancemaento

In [76]:
for campo in xy_train.columns:
    if xy_train[campo].dtype in  ['int64', 'int32', 'float64', 'float32']:
        fig = make_subplots(rows=1, cols=2, subplot_titles=[f'Histograma de {campo}', f'Box Plot de {campo}'])
        fig.add_trace(
            px.histogram(xy_train, x=campo, histnorm="percent", nbins=60).data[0],
            row=1, col=1
        )

        fig.add_trace(
            px.box(xy_train, y=campo).data[0],
            row=1, col=2
            )
        fig.update_layout(title_text=f'{campo}', showlegend=False)
        fig.show()
    else:
        fig = px.histogram(xy_train, x=campo, histnorm="percent", nbins=60)
        fig.update_layout(title_text=f'{campo}', showlegend=False)
        fig.show()


Survived parece bem equilibrada, especialmente se for considerada  origem dos dados.
as variaveis não apresentam comportamento estranho nem desbalanceado: 
pclasse


a idade tem um pico em 25, o que pode parecer estranjho, mas faz sentido se considerado o grande numero de publico jovem e masculino, tanto de trabalhadores do próprio navio, como trabalhadores de class mais baixas.

algumas idades mais elevadas se destacam, mas todas estão dentro de um intervalo razoável

considero que nenhum outlier, todos os valores, mesmo os que se destacam do padrão, estão dentro do comṕortamento esperado razoável.
muitas classes desbalanceadas, vamos aplicar o um balancemaneto

In [78]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin',
       'Embarked_C', 'Embarked_Q', 'Embarked_S']



In [ ]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
x, y = smote.fit_resample(xy_train[features], xy_train['Survived'])
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)
x_test_scaled = scaler.transform(x_test)

**2. Aperfeiçoamento:**
**Aplique as técnicas que aprendemos para melhorar o desempenho do seu modelo:**

**Hiperparâmetros:** Utilize GridSearchCV ou RandomSearchCV para encontrar os melhores parâmetros.

**Cross Validation:** Avalie a robustez do modelo utilizando validação cruzada para garantir que ele generaliza bem.

**Balanceamento de Classes:** Se o seu modelo lida com problemas de classes desbalanceadas, explore técnicas como SMOTE, undersampling ou oversampling.

**Padronização e Normalização:** Lembre-se de padronizar os dados, especialmente se for usar modelos que são sensíveis à escala das variáveis.

In [83]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

## XBoost

In [105]:
param_grid_xgboost = {
    'max_depth': [3, 5, 7],              
    'n_estimators': [50, 100, 200],     
    'learning_rate': [0.01, 0.1, 0.2],   
    'subsample': [0.8, 1.0],           
    'colsample_bytree': [0.8, 1.0]       
}

xgboost = xgb.XGBClassifier()

grid_search_xgboost = GridSearchCV(
    estimator = xgboost,   
    param_grid=param_grid_xgboost,     
    scoring='accuracy',         
    cv=5,                       
    n_jobs=-1                  
)

grid_search_xgboost.fit(x_scaled, y)

print("Melhores Parâmetros:", grid_search_xgboost.best_params_)
print("Melhor Acurácia:", grid_search_xgboost.best_score_)

best_model_xgboost = grid_search_xgboost.best_estimator_
y_pred_xgboost = best_model_xgboost.predict(x_test_scaled)

y_pred_train = best_model_xgboost.predict(x_scaled)
accuracy = accuracy_score(y, y_pred_train)
print(f"Acurácia: {accuracy:.2f}")

report = classification_report(y, y_pred_train)
print("Relatório de Classificação:")
print(report)

Melhores Parâmetros: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'subsample': 1.0}
Melhor Acurácia: 0.8571274387712744
Acurácia: 0.92
Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.90      0.95      0.92       549
           1       0.95      0.90      0.92       549

    accuracy                           0.92      1098
   macro avg       0.92      0.92      0.92      1098
weighted avg       0.92      0.92      0.92      1098



## Random forest

In [104]:
# Hyperparametros
params_grid_rf = {
    'n_estimators': randint(50, 200),
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': randint(2, 10)
}

randomforest = RandomForestClassifier(random_state=42)

rf_search = RandomizedSearchCV(
    estimator= randomforest,
    param_distributions= params_grid_rf,
    n_iter=100,  
    random_state=42,
    cv = 5,
    n_jobs=-1  )

rf_search.fit(x_scaled, y)

best_params_rf = rf_search.best_params_
print("Melhores Parâmetros Encontrados:", best_params_rf)

y_pred_train = rf_search.predict(x_scaled)
accuracy = accuracy_score(y, y_pred_train)
print(f"Acurácia: {accuracy:.2f}")

report = classification_report(y, y_pred_train)
print("Relatório de Classificação:")
print(report)

y_pred_rf = rf_search.predict(x_test_scaled)


Melhores Parâmetros Encontrados: {'max_depth': 20, 'min_samples_split': 4, 'n_estimators': 82}
Acurácia: 0.97
Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       549
           1       0.98      0.96      0.97       549

    accuracy                           0.97      1098
   macro avg       0.97      0.97      0.97      1098
weighted avg       0.97      0.97      0.97      1098



# Travei na parte de como tem que exportar, vou ter que ver algum video; tbm preciso revisar a escrita do projeto

In [113]:
df_rf = pd.DataFrame()
df_rf['Survived'] = y_pred_rf

df_xb = pd.DataFrame()
df_xb['Survived'] = y_pred_xgboost

In [115]:
np.savetxt('y_pred_rf.csv', df_rf, delimiter=',', fmt='%d')
np.savetxt('y_pred_xboost.csv', df_xb , delimiter=',', fmt='%d')

**3. Submissão no Kaggle:**
Treine o seu modelo com os dados de treino e gere as previsões para os dados de teste. Lembre-se de que o conjunto de teste não possui a variável alvo (y_test), pois a avaliação será feita com base nas submissões no Kaggle.
Submeta suas previsões na competição do Titanic no Kaggle.

**4. Entrega:**
Envie o código que você desenvolveu, detalhando cada etapa do seu processo de modelagem, explicando as escolhas feitas e como essas ajudaram a melhorar o modelo.

Junto com o código, envie um print do seu score obtido na plataforma do Kaggle. Esse score será a sua métrica final de avaliação, mostrando como o seu modelo se compara com os demais.

**5. Competição Saudável:**
A ideia é trazer um senso de competição saudável, então não vale replicar exatamente o que fizemos na aula! Inove, explore novas combinações de parâmetros e técnicas, e mostre do que é capaz. O importante é exercitar o pensamento crítico e a capacidade de experimentar.

**Dicas Finais:**

Seja criativo e tenha um olhar crítico sobre o que pode ser melhorado.
Teste diferentes abordagens e não se prenda a um único caminho.
Lembre-se de que, mais do que alcançar o melhor score, o objetivo é aprender e aplicar o conhecimento de forma prática e eficaz.
Boa sorte! Estamos ansiosos para ver como cada um de vocês vai se sair nesse desafio e quais insights irão surgir dessa competição!

Ao final dessa atividade vocês terão participado da primeira competição publica de ciência de dados de vocês = )


